In [ ]:


import cv2 as cv
import numpy as np

# ─────────────────────────────────────────────
#  Load images
# ─────────────────────────────────────────────
im1 = cv.imread('c1.jpg', cv.IMREAD_REDUCED_COLOR_4)
im2 = cv.imread('c2.jpg', cv.IMREAD_REDUCED_COLOR_4)
assert im1 is not None and im2 is not None, "Images not found – check path."

DISPLAY_W, DISPLAY_H = 900, 650


# PART (a)  Manual point selection
N = 6        
global n
n = 0
p1 = np.empty((N, 2))
p2 = np.empty((N, 2))


def draw_circle(event, x, y, flags, param):
    global n
    pts, canvas = param
    if event == cv.EVENT_LBUTTONDOWN:
        cv.circle(canvas, (x, y), 6, (0, 0, 255), -1)
        cv.putText(canvas, str(n + 1), (x + 8, y - 8),
                   cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        pts[n] = (x, y)
        n += 1


# --- Collect points in Image 1 ---
im1copy = im1.copy()
cv.namedWindow('Image 1 – click 6 pts', cv.WINDOW_AUTOSIZE)
cv.setMouseCallback('Image 1 – click 6 pts', draw_circle, [p1, im1copy])
print(f"Click {N} corresponding points on Image 1, then press any key.")
while True:
    cv.imshow('Image 1 – click 6 pts', im1copy)
    if n >= N:
        break
    if cv.waitKey(20) & 0xFF == 27:
        break
cv.destroyAllWindows()

# --- Collect points in Image 2 ---
n = 0
im2copy = im2.copy()
cv.namedWindow('Image 2 – click 6 pts', cv.WINDOW_AUTOSIZE)
cv.setMouseCallback('Image 2 – click 6 pts', draw_circle, [p2, im2copy])
print(f"Click the SAME {N} corresponding points on Image 2, then press any key.")
while True:
    cv.imshow('Image 2 – click 6 pts', im2copy)
    if n >= N:
        break
    if cv.waitKey(20) & 0xFF == 27:
        break
cv.destroyAllWindows()

print("Points in im1:\n", p1)
print("Points in im2:\n", p2)

# --- Compute homography (maps im1 → frame of im2) ---
H_manual, mask_manual = cv.findHomography(p1, p2, method=cv.RANSAC)
print("\nManual Homography H:\n", H_manual)

# --- Warp im1 into im2's frame ---
h2, w2 = im2.shape[:2]
im1_warped_manual = cv.warpPerspective(im1, H_manual, (w2, h2))

# Display
cv.namedWindow('(a) im1 warped (manual)', cv.WINDOW_AUTOSIZE)
cv.imshow('(a) im1 warped (manual)', im1_warped_manual)
cv.namedWindow('(a) im2 reference', cv.WINDOW_AUTOSIZE)
cv.imshow('(a) im2 reference', im2)
cv.waitKey(0)
cv.destroyAllWindows()

# ═══════════════════════════════════════════════════════════════════════════
# PART (b)  Difference image (manual homography)
# ═══════════════════════════════════════════════════════════════════════════

# Paste im2 into the warped canvas (same region), then subtract
diff_manual = cv.absdiff(im1_warped_manual, im2)

# Amplify for visibility
diff_manual_bright = cv.convertScaleAbs(diff_manual, alpha=3.0, beta=0)

cv.namedWindow('(b) Difference – manual H', cv.WINDOW_AUTOSIZE)
cv.imshow('(b) Difference – manual H', diff_manual_bright)
cv.waitKey(0)
cv.destroyAllWindows()

cv.imwrite('part_a_warped_manual.jpg',  im1_warped_manual)
cv.imwrite('part_b_diff_manual.jpg',   diff_manual_bright)
print("[Saved] part_a_warped_manual.jpg, part_b_diff_manual.jpg")

# 
# PART (c)  SIFT keypoints, descriptors, and matches display
#

# Convert to grayscale for feature detection
gray1 = cv.cvtColor(im1, cv.COLOR_BGR2GRAY)
gray2 = cv.cvtColor(im2, cv.COLOR_BGR2GRAY)

# Create SIFT detector (fall back to ORB if SIFT unavailable)
try:
    detector = cv.SIFT_create()
    use_sift = True
    print("\nUsing SIFT for feature detection.")
except AttributeError:
    detector = cv.ORB_create(nfeatures=2000)
    use_sift = False
    print("\nSIFT unavailable – falling back to ORB.")

kp1, des1 = detector.detectAndCompute(gray1, None)
kp2, des2 = detector.detectAndCompute(gray2, None)
print(f"Keypoints: im1={len(kp1)}, im2={len(kp2)}")

# Match descriptors
if use_sift:
    # FLANN-based matcher for SIFT (float descriptors)
    FLANN_INDEX_KDTREE = 1
    index_params  = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
    search_params = dict(checks=50)
    matcher = cv.FlannBasedMatcher(index_params, search_params)
    matches_knn = matcher.knnMatch(des1, des2, k=2)
    # Lowe's ratio test
    good_matches = [m for m, n_m in matches_knn if m.distance < 0.75 * n_m.distance]
else:
    # BFMatcher with Hamming distance for ORB (binary descriptors)
    bf = cv.BFMatcher(cv.NORM_HAMMING, crossCheck=False)
    matches_knn = bf.knnMatch(des1, des2, k=2)
    good_matches = [m for m, n_m in matches_knn if m.distance < 0.75 * n_m.distance]

print(f"Good matches after ratio test: {len(good_matches)}")

# Draw top-100 matches
draw_params = dict(
    matchColor=(0, 255, 0),
    singlePointColor=(255, 0, 0),
    flags=cv.DrawMatchesFlags_DEFAULT
)
match_img = cv.drawMatches(im1, kp1, im2, kp2,
                            good_matches[:100], None, **draw_params)

cv.namedWindow('(c) Feature Matches', cv.WINDOW_AUTOSIZE)
cv.imshow('(c) Feature Matches', match_img)
cv.waitKey(0)
cv.destroyAllWindows()

cv.imwrite('part_c_matches.jpg', match_img)
print("[Saved] part_c_matches.jpg")

# ═══════════════════════════════════════════════════════════════════════════
# PART (d)  Homography from feature matches → Warp → Difference → Compare
# ═══════════════════════════════════════════════════════════════════════════

if len(good_matches) < 4:
    print("Not enough matches to compute homography!")
else:
    src_pts = np.float32([kp1[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)

    H_auto, mask_auto = cv.findHomography(src_pts, dst_pts, cv.RANSAC, ransacReprojThreshold=5.0)
    inliers = int(mask_auto.sum())
    print(f"\nAutomatic Homography H (RANSAC inliers: {inliers}/{len(good_matches)}):\n", H_auto)

    # Warp im1 into im2's frame using automatic H
    im1_warped_auto = cv.warpPerspective(im1, H_auto, (w2, h2))

    # Difference image
    diff_auto       = cv.absdiff(im1_warped_auto, im2)
    diff_auto_bright = cv.convertScaleAbs(diff_auto, alpha=3.0, beta=0)

    # Display side-by-side comparison
    cv.namedWindow('(d) im1 warped (auto)', cv.WINDOW_AUTOSIZE)
    cv.imshow('(d) im1 warped (auto)', im1_warped_auto)
    cv.namedWindow('(d) Difference – auto H', cv.WINDOW_AUTOSIZE)
    cv.imshow('(d) Difference – auto H', diff_auto_bright)
    cv.waitKey(0)
    cv.destroyAllWindows()

    cv.imwrite('part_d_warped_auto.jpg', im1_warped_auto)
    cv.imwrite('part_d_diff_auto.jpg',  diff_auto_bright)
    print("[Saved] part_d_warped_auto.jpg, part_d_diff_auto.jpg")

    # ── Quantitative comparison ─────────────────────────────────────────────
    mean_diff_manual = np.mean(diff_manual)
    mean_diff_auto   = np.mean(diff_auto)
    print("\n── Quantitative Comparison ──────────────────────────────")
    print(f"  Mean pixel difference (manual H): {mean_diff_manual:.3f}")
    print(f"  Mean pixel difference (auto   H): {mean_diff_auto:.3f}")
    print("─────────────────────────────────────────────────────────")

"""
── Discussion (Part d) ─────────────────────────────────────────────────────

Manual vs. Automatic Homography:

1. Accuracy
    Manual clicks (6 points) are discrete and made with a mouse and prone to error .
     The warp could be slightly off, resulting in a greater mean
     difference even if the boards are the same.
    SIFT/ORB with RANSAC uses hundreds inlier matches throughout
     the entire image. The overdetermined system produces a more correct H,
     usually corresponding to a smaller mean difference on identical regions.

2. Robustness
    Manual selection is dependent on choice of 6 points.  Choosing points
     are close together, you don't extrapolate well.
    Feature matching provides correspondence over the entire
     image and hence promotes a better overall alignment.

3. Difference image quality
    Manual H - the difference image looks like it has spurious edges / gradients
     from a slight misalignment, as well as differences between boards.
    With automatic H, grading is more precise so true differences (solders,
     stuff, solder, traces) are more visible with a dark background.
     background.

4. Failure modes
    SIFT/ORB can be unreliable if the images lack texture or have large
     user selection is a good back-up.
    If there are large differences in scale or rotation (> 45°) in the
     may rule out too many points; reduce the test threshold or increase the number of points.

Conclusion: Using the automatic method is generally more accurate
and a clearer difference image, and more appropriate for detecting defects
between the two circuit boards.
────────────────────────────────────────────────────────────────────────────
"""

Click 6 corresponding points on Image 1, then press any key.
Click the SAME 6 corresponding points on Image 2, then press any key.
Points in im1:
 [[265.  44.]
 [469.  95.]
 [323. 639.]
 [ 64. 541.]
 [235. 204.]
 [246. 166.]]
Points in im2:
 [[344.  14.]
 [530. 120.]
 [246. 604.]
 [ 20. 444.]
 [275. 161.]
 [297. 130.]]

Manual Homography H:
 [[ 9.88684576e-01 -2.65152913e-01  1.00144980e+02]
 [ 2.75019192e-01  9.78239270e-01 -1.00056186e+02]
 [ 3.35308758e-05  8.52742191e-06  1.00000000e+00]]
[Saved] part_a_warped_manual.jpg, part_b_diff_manual.jpg

Using SIFT for feature detection.
Keypoints: im1=1439, im2=1489
Good matches after ratio test: 1031
[Saved] part_c_matches.jpg

Automatic Homography H (RANSAC inliers: 996/1031):
 [[ 9.64668600e-01 -2.63855738e-01  1.01516060e+02]
 [ 2.63866591e-01  9.64688017e-01 -9.60490006e+01]
 [ 2.24097551e-07  1.41707284e-08  1.00000000e+00]]
[Saved] part_d_warped_auto.jpg, part_d_diff_auto.jpg

── Quantitative Comparison ─────────────────────────────

'\n── Discussion (Part d) ─────────────────────────────────────────────────────\n\nManual vs. Automatic Homography:\n\n1. Accuracy\n   - Manual clicks (6 points) are coarse and subject to human error (±5-10 px).\n     The resulting warp may have slight misalignment, producing a larger mean\n     difference even where the boards are identical.\n   - SIFT/ORB with RANSAC uses hundreds of inlier correspondences spread across\n     the entire image. This overdetermined system yields a more accurate H,\n     typically giving a smaller mean difference on identical regions.\n\n2. Robustness\n   - Manual selection is sensitive to which 6 points you pick.  Choosing points\n     clustered in one area leads to poor extrapolation elsewhere.\n   - Feature-based matching naturally distributes correspondences over the whole\n     image, improving global alignment.\n\n3. Difference image quality\n   - With manual H, the difference image may show spurious edges / gradients\n     from slight misalignmen